In [1]:
import pandas as pd
import numpy as np
import sklearn
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    root_mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
import time

In [3]:
# Check CUDA installation and version
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Nov__7_19:25:04_Pacific_Standard_Time_2025
Cuda compilation tools, release 13.1, V13.1.80
Build cuda_13.1.r13.1/compiler.36836380_0


In [4]:
# Check for GPU access
!nvidia-smi

Mon Jul  6 15:04:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 576.28                 Driver Version: 576.28         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   39C    P8             11W /  105W |     280MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [9]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing()
X, y = data.data, data.target

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
print(f"X_train.shape:{X_train.shape}\nX_test.shape:{X_train.shape}")

X_train.shape:(16512, 8)
X_test.shape:(16512, 8)


In [12]:
# Baseline model
model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
mse_model = mean_squared_error(y_test, y_pred)
rmse_model = root_mean_squared_error(y_test, y_pred)
mae_model = mean_absolute_error(y_test, y_pred)
r2_model = r2_score(y_test, y_pred)

print(f"MSE :  {mse_model:.4f}")
print(f"RMSE:  {rmse_model:.4f}")
print(f"MAE :  {mae_model:.4f}")
print(f"R²  :  {r2_model:.4f}")

MSE :  0.2226
RMSE:  0.4718
MAE :  0.3096
R²  :  0.8301


In [13]:
# Objective function for Optuna
def objective(trial):

    params = {
        "objective": "reg:squarederror",
        "random_state": 42,

        # Boosting
        "learning_rate": trial.suggest_float("learning_rate",0.01, 0.15, log=True,),

        "n_estimators": trial.suggest_int("n_estimators",200,3000,),

        # Tree Complexity
        "max_depth": trial.suggest_int("max_depth",3,8, ),

        "min_child_weight": trial.suggest_int("min_child_weight",1,15,),

        "gamma": trial.suggest_float("gamma",0,3,),

        # Randomness
        "subsample": trial.suggest_float("subsample",0.6,1.0,),

        "colsample_bytree": trial.suggest_float("colsample_bytree",0.6,1.0,),

        # Regularization
        "reg_alpha": trial.suggest_float("reg_alpha",1e-4,10,log=True,),

        "reg_lambda": trial.suggest_float("reg_lambda",1e-3,20,log=True,),
    }

    model = xgb.XGBRegressor(**params,tree_method= "hist",verbosity = 0,device="cuda")

    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    ).mean()

    return score

sampler = optuna.samplers.TPESampler(
    seed=42,
    multivariate=True,
    n_startup_trials=20,
    n_ei_candidates=48,
)

study = optuna.create_study(
    study_name="objective",
    direction="maximize",
    sampler=sampler,
)

C:\.venv\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(


In [ ]:
start_time = time.time()
study.optimize(objective, n_trials=500, show_progress_bar=True, n_jobs= 1)
end_time = time.time()
elapsed_time = end_time - start_time

print(f"Elapsed Time: {elapsed_time:.4f} seconds")

  0%|          | 0/500 [00:00<?, ?it/s]

[W 2026-07-06 14:50:59,107] Trial 244 failed with parameters: {'learning_rate': 0.014434840920252255, 'n_estimators': 2965, 'max_depth': 4, 'min_child_weight': 7, 'gamma': 0.48362209085627417, 'subsample': 0.6222872979193971, 'colsample_bytree': 0.9937736270209057, 'reg_alpha': 0.00183231098363437, 'reg_lambda': 12.275001834324058} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\.venv\lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\SL\AppData\Local\Temp\ipykernel_17336\3830479795.py", line 33, in objective
    score = cross_val_score(
  File "C:\.venv\lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
  File "C:\.venv\lib\site-packages\sklearn\model_selection\_validation.py", line 677, in cross_val_score
    cv_results = cross_validate(
  File "C:\.venv\lib\site-packages\sklearn\utils\_param_validation.py",